In [1]:
import numpy as np
import pandas as pd

***One Hot Encoding***

In [2]:
labels = np.array(["cat", "dog", "mat", "rat", "sat", "cat", "hat", "fat", "hat", "sat"]).reshape(-1, 1)
labels.flatten()

array(['cat', 'dog', 'mat', 'rat', 'sat', 'cat', 'hat', 'fat', 'hat',
       'sat'], dtype='<U3')

In [3]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False)
builtin_encodings = ohe.fit_transform(labels)

def onehot(labels):
    encodings = []
    sorted_labels = set(np.unique(labels))
    sorted_labels = list(sorted(sorted_labels))
    total_labels = len(sorted_labels)

    for label in labels:
        encoding = [0] * total_labels
        index = sorted_labels.index(label)
        encoding[index] = 1
        encodings.append(encoding)
    
    return np.array(encodings)

custom_encodings = onehot(labels)
builtin_encodings, custom_encodings

(array([[1., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 0., 0., 1.],
        [1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 1.]]),
 array([[1, 0, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, 0, 1],
        [1, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0, 0],
        [0, 0, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 1]]))

***Bag of Words (BoW)***

In [4]:
statements = np.array([
    "cat catch the mat that catch hat",
    "rat match hat dog catch hat",
    "sat on the dog cat sat on rat",
    "dog catch cat dog fat fat",
    "fat rat run match rat"
])
statements

array(['cat catch the mat that catch hat', 'rat match hat dog catch hat',
       'sat on the dog cat sat on rat', 'dog catch cat dog fat fat',
       'fat rat run match rat'], dtype='<U32')

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer()
builtin_bow = bow.fit_transform(statements).toarray()

def bow(statements):
    vocabulary = set()
    for statement in statements:
        words = statement.split(' ')
        for word in words:
            vocabulary.add(word)

    vocabulary = sorted(vocabulary)
    vocab_size = len(vocabulary)

    encodings = []
    for statement in statements:
        encoding = [0] * vocab_size
        words = statement.split(' ')
        for word in words:
            encoding[ vocabulary.index(word) ] += 1
        encodings.append(encoding)
    
    return encodings

custom_bow = bow(statements)
builtin_bow, custom_bow

(array([[1, 2, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1],
        [0, 1, 1, 0, 2, 0, 1, 0, 1, 0, 0, 0, 0],
        [1, 0, 1, 0, 0, 0, 0, 2, 1, 0, 2, 0, 1],
        [1, 1, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0, 1, 0, 2, 1, 0, 0, 0]]),
 [[1, 2, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1],
  [0, 1, 1, 0, 2, 0, 1, 0, 1, 0, 0, 0, 0],
  [1, 0, 1, 0, 0, 0, 0, 2, 1, 0, 2, 0, 1],
  [1, 1, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 1, 0, 0, 1, 0, 2, 1, 0, 0, 0]])

***TF-IDF: Term Frequency - Inverse Document Frequency***

- **Term Frequency:** Word Frequency in Document / Total Words in Document
- **Inverse Document Frequency:** log(Total Documents / Number of Document in which word exist)

In [6]:
documents = np.array([
    "The cat sits on the mat",
    "The dog sits on the rug",
    "The cat and dog play together",
    "Dogs and cats are common pets",
    "The mat is clean and comfortable"
])

for i in range(len(documents)):
    documents[i] = documents[i].lower()

documents

array(['the cat sits on the mat', 'the dog sits on the rug',
       'the cat and dog play together', 'dogs and cats are common pets',
       'the mat is clean and comfortable'], dtype='<U32')

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
builtin_probs = tfidf.fit_transform(documents).toarray()


def create_vocabulary(documents):
    vocabulary = set()
    for document in documents:
        words = document.lower().split(' ')
        for word in words:
            vocabulary.add(word)
    
    return sorted(vocabulary)

def term_frequency(documents, vocabulary):
    tfs = []
    vocab_size = len(vocabulary)
    
    for document in documents:
        probs = [0] * vocab_size
        words = document.split(' ')
        for word in words:
            probs[ vocabulary.index(word) ] += 1
        
        probs = np.array(probs)
        probs = probs / len(words)
        tfs.append(probs)

    return tfs


def inverse_document_frequency(documents, vocabulary):
    vocabs = {word: 0 for word in vocabulary}
    for document in documents:
        words = set(document.split(' '))
        for word in words:
            vocabs[word] += 1

    vocab_size = len(vocabulary)
    idfs = np.zeros((len(documents), vocab_size))

    for document_index in range(len(idfs)):
        for index in range(vocab_size):
            idfs[document_index, index] = float(np.log(len(documents) / vocabs[vocabulary[index]]) )

    return np.array(idfs)
    

def tfidf(documents):
    vocabulary = create_vocabulary(documents)
    vocab_size = len(vocabulary)
    tfs = term_frequency(documents, vocabulary)
    idfs = inverse_document_frequency(documents, vocabulary)

    tfidfs = np.zeros((len(documents), vocab_size))
    for document_index in range(len(documents)):
        for index in range(vocab_size):
            tfidfs[document_index, index] = tfs[document_index][index] * idfs[document_index][index]

    return tfidfs

builtin_probs, tfidf(documents)

(array([[0.        , 0.        , 0.40994345, 0.        , 0.        ,
         0.        , 0.        , 0.        , 0.        , 0.        ,
         0.40994345, 0.40994345, 0.        , 0.        , 0.        ,
         0.40994345, 0.57252551, 0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        , 0.        , 0.39263182, 0.        , 0.        ,
         0.        , 0.39263182, 0.        , 0.        , 0.48665702,
         0.39263182, 0.54834816, 0.        ],
        [0.33205571, 0.        , 0.40002359, 0.        , 0.        ,
         0.        , 0.        , 0.40002359, 0.        , 0.        ,
         0.        , 0.        , 0.        , 0.49581892, 0.        ,
         0.        , 0.27933573, 0.49581892],
        [0.28691208, 0.42841136, 0.        , 0.42841136, 0.        ,
         0.        , 0.42841136, 0.        , 0.42841136, 0.        ,
         0.        , 0.        , 0.42841136, 0.        , 0.        ,
         0.        , 0.        , 0

***N grams***

In [12]:
corpus = np.array([
    "I like machine learning",
    "I like deep learning",
    "I enjoy machine learning",
    "Machine learning is useful",
    "Deep learning is useful"
])
corpus

array(['I like machine learning', 'I like deep learning',
       'I enjoy machine learning', 'Machine learning is useful',
       'Deep learning is useful'], dtype='<U26')

In [23]:
from collections import Counter

def generate_ngrams(corpus, n):
    ngrams = []
    for document in corpus:
        words = ["<s>"] * (n-1) + document.split(' ') + ["</s>"] * (n-1)
        for i in range(len(words) - n + 1):
            ngrams.append(tuple(words[i:i+n]))

    return Counter(ngrams)

unigrams = generate_ngrams(corpus, 1)
bigrams = generate_ngrams(corpus, 2)
trigrams = generate_ngrams(corpus, 3)

In [24]:
def unigram_to_probabilities(unigrams):
    total_words = sum(unigrams.values())
    probabilities = {}
    for word in unigrams:
        probabilities[word] = unigrams[word] / total_words
    return probabilities
    
unigram_probability = unigram_to_probabilities(unigrams)
unigram_probability

{('I',): 0.15,
 ('like',): 0.1,
 ('machine',): 0.1,
 ('learning',): 0.25,
 ('deep',): 0.05,
 ('enjoy',): 0.05,
 ('Machine',): 0.05,
 ('is',): 0.1,
 ('useful',): 0.1,
 ('Deep',): 0.05}

In [59]:
def bigrams_to_probabilities(bigrams, unigrams):
    probabilities = {}
    for words in bigrams:
        first_word = (words[0], )
        probabilities[words] = bigrams[words] / max(1, unigrams[first_word])
    return probabilities
    
bigram_probability = bigrams_to_probabilities(bigrams, unigrams)
bigram_probability

{('<s>', 'I'): 3.0,
 ('I', 'like'): 0.6666666666666666,
 ('like', 'machine'): 0.5,
 ('machine', 'learning'): 1.0,
 ('learning', '</s>'): 0.6,
 ('like', 'deep'): 0.5,
 ('deep', 'learning'): 1.0,
 ('I', 'enjoy'): 0.3333333333333333,
 ('enjoy', 'machine'): 1.0,
 ('<s>', 'Machine'): 1.0,
 ('Machine', 'learning'): 1.0,
 ('learning', 'is'): 0.4,
 ('is', 'useful'): 1.0,
 ('useful', '</s>'): 1.0,
 ('<s>', 'Deep'): 1.0,
 ('Deep', 'learning'): 1.0}

In [62]:
def trigrams_to_probabilities(trigrams, bigrams):
    probabilities = {}
    for words in trigrams:
        prior_words = words[:2]
        probabilities[words] = trigrams[words] / max(1, bigrams[prior_words])
    return probabilities

trigram_probability = trigrams_to_probabilities(trigrams, bigrams)
trigram_probability

{('<s>', '<s>', 'I'): 3.0,
 ('<s>', 'I', 'like'): 0.6666666666666666,
 ('I', 'like', 'machine'): 0.5,
 ('like', 'machine', 'learning'): 1.0,
 ('machine', 'learning', '</s>'): 1.0,
 ('learning', '</s>', '</s>'): 1.0,
 ('I', 'like', 'deep'): 0.5,
 ('like', 'deep', 'learning'): 1.0,
 ('deep', 'learning', '</s>'): 1.0,
 ('<s>', 'I', 'enjoy'): 0.3333333333333333,
 ('I', 'enjoy', 'machine'): 1.0,
 ('enjoy', 'machine', 'learning'): 1.0,
 ('<s>', '<s>', 'Machine'): 1.0,
 ('<s>', 'Machine', 'learning'): 1.0,
 ('Machine', 'learning', 'is'): 1.0,
 ('learning', 'is', 'useful'): 1.0,
 ('is', 'useful', '</s>'): 1.0,
 ('useful', '</s>', '</s>'): 1.0,
 ('<s>', '<s>', 'Deep'): 1.0,
 ('<s>', 'Deep', 'learning'): 1.0,
 ('Deep', 'learning', 'is'): 1.0}

In [64]:
test_statement = "I like"
def predict_next(trigram_probability, test_statement):
    test_statement = tuple(test_statement.split(' '))
    predictions = {}
    for keys in trigram_probability:
        if keys[:2] == test_statement:
            predictions[keys[2]] = trigram_probability[keys]
    return predictions

predict_next(trigram_probability, test_statement)

{'machine': 0.5, 'deep': 0.5}